# 1. Import Necessary Packages

In [ ]:
# Uncomment and run the following line if required packages are not installed
# !pip install pandas numpy scipy statsmodels matplotlib seaborn openpyxl

# Core data handling
import pandas as pd        # Tabular data structures (DataFrame), I/O, groupby/merge, etc.
import numpy as np         # Numerical computing: arrays, vectorization, stats helpers

# Plotting
import matplotlib.pyplot as plt  # Base plotting (figures, axes, annotations, savefig)
import seaborn as sns            # Statistical/beautiful plots built on matplotlib (themes, high-level charts)

# File & paths
import os                        # File system ops (paths, mkdir, exists), environment variables

# Statistics / tests
from scipy.stats import mannwhitneyu
from scipy import stats 

# Others
from matplotlib.backends.backend_pdf import PdfPages  # Save multiple figures into a single PDF
import matplotlib.image as mpimg                      # Read/display images (e.g., when assembling figure grids)

import matplotlib as mpl


# 2. Set up

## 2.1 Set Up Working Path

In [ ]:
os.getcwd()
# Set global paths
PROJECT_PATH = r"C:\Your\Replication\Folder"

# subfolders
DATA_PATH   = os.path.join(PROJECT_PATH, "Data")
RESULT_PATH = os.path.join(PROJECT_PATH, "Result")

# Print paths
print("Project Path:", PROJECT_PATH)
print("Data Path:", DATA_PATH)
print("Result Path:", RESULT_PATH)

# Optionally change the working directory to one of them
os.chdir(DATA_PATH)
#os.chdir(RESULT_PATH)

## 2.2 Set Up Formatting

In [ ]:
# Apply Nature-style formatting globally
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'figure.dpi': 600,
})

# Confirm update
print("Matplotlib parameters updated to Nature Food journal style.")

# Set font sizes for readability
plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 14,
    'legend.fontsize': 10,
    'figure.titlesize': 8
})


## 2.3 Load Dataset

In [ ]:
os.chdir(DATA_PATH)
# Load the dataset
df = pd.read_excel("GNPD-AllFoodDrink_Claim_NPMScore_2015_2024_synthetic.xlsx", engine="openpyxl")

# 3. Results

## 3.1 Tables

### Table 1. The Healthfulness of New Products by Category

In [ ]:
# If needed, install python-docx once:
# !pip install python-docx

from docx import Document
from docx.shared import Inches
from docx.enum.section import WD_ORIENT
import os
import pandas as pd

# ------------------------------------------------------------
# 1. Construct Less healthy (%) under NPM
#    Food: np_score >= 4
#    Beverages: np_score >= 1
# ------------------------------------------------------------
df = df.copy()

df["less_healthy_npm"] = pd.NA
df.loc[df["NewCategory"] == "Beverages", "less_healthy_npm"] = (
    pd.to_numeric(df.loc[df["NewCategory"] == "Beverages", "np_score"], errors="coerce") >= 1
).astype(float)
df.loc[df["NewCategory"] != "Beverages", "less_healthy_npm"] = (
    pd.to_numeric(df.loc[df["NewCategory"] != "Beverages", "np_score"], errors="coerce") >= 4
).astype(float)

# Your list of indicators
Health_indicators = [
    "np_score", "less_healthy_npm",
    "total_a_points", "energy_points", "sat_fat_points",
    "total_sugar_points", "sodium_points",
    "total_c_points", "adjusted_c_points",
    "fvn_points", "fiber_points", "protein_points"
]

# Ensure numeric
df[Health_indicators] = df[Health_indicators].apply(pd.to_numeric, errors='coerce')

# ------------------------------------------------------------
# 2. Group and compute
# ------------------------------------------------------------
grp       = df.groupby("NewCategory")
count_ser = grp.size()
mean_df   = grp[Health_indicators].mean()
std_df    = grp[Health_indicators].std()

# Format mean (std)
formatted = pd.DataFrame(index=mean_df.index)
for col in Health_indicators:
    if col == "less_healthy_npm":
        # convert from share to percent
        formatted[col] = mean_df[col].combine(
            std_df[col],
            lambda m, s: f"{m * 100:.2f} ({s * 100:.2f})" if pd.notna(m) else ""
        )
    else:
        formatted[col] = mean_df[col].combine(
            std_df[col],
            lambda m, s: f"{m:.2f} ({s:.2f})" if pd.notna(m) else ""
        )

# Build summary_df
summary_df = formatted.copy()
summary_df.insert(0, "Number of New Products", count_ser)
summary_df = summary_df.reset_index().rename(columns={"NewCategory": "Category"})

# Add overall row
overall = {
    "Category": "All products",
    "Number of New Products": len(df)
}
for col in Health_indicators:
    m, s = df[col].mean(), df[col].std()
    if col == "less_healthy_npm":
        overall[col] = f"{m * 100:.2f} ({s * 100:.2f})"
    else:
        overall[col] = f"{m:.2f} ({s:.2f})"

summary_df = pd.concat(
    [summary_df, pd.DataFrame([overall])],
    ignore_index=True
)

# ------------------------------------------------------------
# 3. Export into a .docx in landscape with table grid and merged headers
# ------------------------------------------------------------
doc = Document()
sec = doc.sections[-1]
sec.orientation = WD_ORIENT.LANDSCAPE
sec.page_width, sec.page_height = sec.page_height, sec.page_width

for attr in ("left_margin", "right_margin", "top_margin", "bottom_margin"):
    setattr(sec, attr, Inches(0.5))

# Map raw column names → second-row subheaders
subhdr = {
    "Category": "Category",
    "Number of New Products": "Number of New Products",
    "np_score": "NP Score",
    "less_healthy_npm": "Less Healthy (%)",
    "total_a_points": "Total A Points",
    "energy_points": "Energy",
    "sat_fat_points": "Saturated Fat",
    "total_sugar_points": "Total Sugar",
    "sodium_points": "Sodium",
    "total_c_points": "Total C Points",
    "adjusted_c_points": "Adjusted Total C Points",
    "fvn_points": "Fruit, Veg & Nuts",
    "fiber_points": "Fiber",
    "protein_points": "Protein"
}

n_rows = summary_df.shape[0] + 2   # 2 header rows + data
n_cols = summary_df.shape[1]
table = doc.add_table(rows=n_rows, cols=n_cols)
table.style = "Table Grid"

# Top-level merges: (sr, sc, er, ec, text)
merges = [
    (0, 0, 1, 0, "Category"),
    (0, 1, 1, 1, "Number of New Products"),
    (0, 2, 1, 2, "NP Score"),
    (0, 3, 1, 3, "Less Healthy Under NPM"),
    (0, 4, 0, 8, "A Points"),
    (0, 9, 0, 13, "C Points"),
]

for sr, sc, er, ec, txt in merges:
    mc = table.cell(sr, sc).merge(table.cell(er, ec))
    mc.text = txt
    mc.paragraphs[0].alignment = 1

# Second-row subheaders
for j, col in enumerate(summary_df.columns):
    cell = table.cell(1, j)
    cell.text = subhdr[col]
    cell.paragraphs[0].alignment = 1

# Fill the data
for i, row in enumerate(summary_df.itertuples(index=False), start=2):
    for j, val in enumerate(row):
        table.cell(i, j).text = str(val)

# ------------------------------------------------------------
# 4. Save out
# ------------------------------------------------------------
os.chdir(RESULT_PATH)

out_docx = "Table1_less_healthy_summary.docx"
out_xlsx = "Table1_less_healthy_summary_by_Category.xlsx"

doc.save(out_docx)
summary_df.to_excel(out_xlsx, index=False)

print("✅ Written:", out_docx)
print("✅ Exported:", out_xlsx)

## 3.2 Figures

### Figure 1 Trends in Claims and Product Healthfulness (2015–2024)

In [ ]:
import matplotlib as mpl

# ========== Nature-style Font Settings ==========
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.size'] = 7
mpl.rcParams['axes.labelsize'] = 7
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 6

# ================== DATA PREP ==================
df = df.copy()

claim_count_columns = ['Num_Minus', 'Num_Plus', 'Num_Natural', 'Num_Functional']
claim_type_labels   = ['Minus Claims', 'Plus Claims', 'Natural Claims', 'Functional Claims']

df['Year'] = df['Year'].astype(int)
df['NewCategory'] = df['NewCategory'].astype(str)
df['np_score'] = pd.to_numeric(df['np_score'], errors='coerce')

# Construct Less Healthy under NPM
df['less_healthy_npm'] = np.where(
    df['NewCategory'] == 'Beverages',
    (df['np_score'] >= 1).astype(int),
    (df['np_score'] >= 4).astype(int)
)

line_data = []

# Claim lines
for col, label in zip(claim_count_columns, claim_type_labels):
    df[f'{col}_has_claim'] = df[col] > 0
    grouped = df.groupby('Year')[f'{col}_has_claim']
    mean = grouped.mean() * 100
    std = grouped.std() * 100
    count = grouped.count()
    ci95 = 1.96 * std / np.sqrt(count)

    for year in mean.index:
        line_data.append({
            'Year': year,
            'Display Label': label,
            'Share (%)': float(f"{mean[year]:.1f}"),
            'Std Dev': float(f"{std[year]:.1f}"),
            'CI Lower': float(f"{mean[year] - ci95[year]:.1f}"),
            'CI Upper': float(f"{mean[year] + ci95[year]:.1f}"),
            'Label': f"{mean[year]:.1f}"
        })

# Any claim line
df['AnyClaim'] = df[claim_count_columns].sum(axis=1) > 0
grouped = df.groupby('Year')['AnyClaim']
mean = grouped.mean() * 100
std = grouped.std() * 100
count = grouped.count()
ci95 = 1.96 * std / np.sqrt(count)

for year in mean.index:
    line_data.append({
        'Year': year,
        'Display Label': 'Any Studied On-Package Claim',
        'Share (%)': float(f"{mean[year]:.1f}"),
        'Std Dev': float(f"{std[year]:.1f}"),
        'CI Lower': float(f"{mean[year] - ci95[year]:.1f}"),
        'CI Upper': float(f"{mean[year] + ci95[year]:.1f}"),
        'Label': f"{mean[year]:.1f}"
    })

# Less healthy bars
grouped = df.groupby('Year')['less_healthy_npm']
mean = grouped.mean() * 100
std = grouped.std() * 100
count = grouped.count()
ci95 = 1.96 * std / np.sqrt(count)

for year in mean.index:
    line_data.append({
        'Year': year,
        'Display Label': 'Less Healthy Under NPM',
        'Share (%)': float(f"{mean[year]:.1f}"),
        'Std Dev': float(f"{std[year]:.1f}"),
        'CI Lower': float(f"{mean[year] - ci95[year]:.1f}"),
        'CI Upper': float(f"{mean[year] + ci95[year]:.1f}"),
        'Label': f"{mean[year]:.1f}"
    })

plot_df_all_augmented = pd.DataFrame(line_data)

# ================== PLOTTING ==================
style_map = {
    'Any Studied On-Package Claim': {'linestyle': '--', 'linewidth': 3.0, 'marker': 'o', 'markersize': 7, 'color': '#55A868'},
    'Minus Claims':              {'linestyle': '-',  'linewidth': 3.0, 'marker': 'x', 'markersize': 8, 'color': '#DD8452'},
    'Plus Claims':               {'linestyle': '-',  'linewidth': 3.0, 'marker': '^', 'markersize': 7, 'color': '#8172B3'},
    'Natural Claims':            {'linestyle': '-',  'linewidth': 3.0, 'marker': 'D', 'markersize': 6, 'color': '#4C72B0'},
    'Functional Claims':         {'linestyle': '-',  'linewidth': 3.0, 'marker': 's', 'markersize': 6, 'color': '#DA8BC3'}
}

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(10, 6))

y_min = max(0, plot_df_all_augmented['CI Lower'].min() - 5)
y_max = min(105, plot_df_all_augmented['CI Upper'].max() + 5)

# Bars for Less Healthy
health = plot_df_all_augmented[
    plot_df_all_augmented['Display Label'] == 'Less Healthy Under NPM'
].sort_values('Year')

ax.bar(
    health['Year'], health['Share (%)'],
    color='#6E6E6E', alpha=0.40, width=0.45,
    label='Less Healthy Under NPM', zorder=0
)

for x, label_text in zip(health['Year'], health['Label']):
    yv = health.loc[health['Year'] == x, 'Share (%)'].values[0]
    ax.text(x, yv + 1.0, label_text, ha='center', va='bottom', fontsize=9, color='#4A4A4A')

# Lines
for series in ['Any Studied On-Package Claim', 'Minus Claims', 'Plus Claims', 'Natural Claims', 'Functional Claims']:
    data = plot_df_all_augmented[plot_df_all_augmented['Display Label'] == series].sort_values('Year')
    st = style_map[series]
    ax.plot(
        data['Year'], data['Share (%)'],
        label=series, **st, zorder=3
    )
    for x, label_text in zip(data['Year'], data['Label']):
        y = data.loc[data['Year'] == x, 'Share (%)'].values[0]
        ax.text(x, y + 1.2, label_text, ha='center', va='bottom', fontsize=9, color=st['color'])

# Labels + axes
ax.set_xlabel("Year")
ax.set_ylabel("Share of Products (%)")
ax.set_xticks(sorted(df['Year'].unique()))
ax.set_ylim(y_min, y_max)

ax.grid(False)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

handles, labels = ax.get_legend_handles_labels()
order = ['Less Healthy Under NPM', 'Any Studied On-Package Claim', 'Minus Claims',
         'Plus Claims', 'Natural Claims', 'Functional Claims']
sorted_handles = [handles[labels.index(l)] for l in order]
ax.legend(
    sorted_handles, order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=3,
    frameon=False
)

plt.tight_layout()

os.chdir(RESULT_PATH)
fig.savefig("Figure1_Trend_of_Claims_and_Less_Healthy.svg", bbox_inches='tight')
fig.savefig("Figure1_Trend_of_Claims_and_Less_Healthy.png", dpi=600, bbox_inches='tight')
fig.savefig("Figure1_Trend_of_Claims_and_Less_Healthy.tiff", dpi=600, bbox_inches='tight')

plt.show()


### Figure 2. Heatmap of Claims Popularity by Product Category and Specific Claims

In [ ]:
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from scipy.stats import binom

# STEP 2: Define claim groups
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

# STEP 3: Human-readable claim labels
claim_vars = [
    'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
    'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
    'LessChol', 'LessGlycemic', 'PlusVitamin', 'HighProtein', 'AddedCalcium',
    'HighFiber',  'AllNatural', 'NoArtAdditives',
    'NoArtColourings', 'NoArtFlavourings', 'NoArtPreservatives',
    'NoAdditivesPreservatives', 'GMOFree', 'Organic', 'Wholegrain',
    'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
    'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
]

claim_labels = [
    'Low/No/Reduced Calorie', 'No Added Sugar', 'Sugar Free', 'Low/Reduced Sugar', 'Diet/Light',
    'Low/No/Reduced Sodium', 'Low/No/Reduced Carb', 'Low/No/Reduced Fat', 'Low/No/Reduced Trans Fat', 'Low/No/Reduced Saturated Fat',
    'Low/No/Reduced Cholesterol', 'Low/No/Reduced Glycemic', 'Vitamin/Mineral Fortified', 'High/Added Protein', 'Added Calcium',
    'High/Added Fiber', 'All Natural Product', 'No Added/Artificial Additives',
    'No Added/Artificial Colourings', 'No Added/Artificial Flavourings', 'No Added/Artificial Preservatives',
    'No Additives/Preservatives', 'GMO Free', 'Organic', 'Whole Grain', 
    'Brain & Nervous System','Immune System', 'Digestive', 'Probiotic/Prebiotic', 
    'Antioxidant', 'Weight & Muscle Gain','Cardiovascular', 'Bone/Skin/Nails&Hair/Eye Health'
]

# STEP 4: Create label map for display
full_label_map = {}
idx = 0
for group, claims in claim_groups.items():
    for claim in claims:
        full_label_map[claim] = f"{group}: {claim_labels[idx]}"
        idx += 1

# STEP 5: Compute share and CI
df_claims = df[['NewCategory'] + claim_vars].copy()
df_claims['NewCategory'] = df_claims['NewCategory'].astype(str)

# Count total products per category
category_counts = df_claims.groupby('NewCategory').size()

# Compute mean and CI
means = df_claims.groupby('NewCategory')[claim_vars].mean()
counts = df_claims.groupby('NewCategory')[claim_vars].sum()

ci_upper = {}
ci_lower = {}
ci_text = pd.DataFrame(index=means.index, columns=means.columns)

for cat in means.index:
    n = category_counts[cat]
    for claim in claim_vars:
        p = means.loc[cat, claim]
        x = counts.loc[cat, claim]
        ci = 1.96 * np.sqrt(p * (1 - p) / n)
        percent = p * 100
        ci_percent = ci * 100
        ci_text.loc[cat, claim] = f"{percent:.1f}±{ci_percent:.1f}"

# STEP 6: Transpose for heatmap
df_transposed = ci_text.T
df_transposed.rename(index=full_label_map, inplace=True)
cleaned_claim_names_y = [label.split(": ", 1)[1] for label in df_transposed.index]
group_names_y = [label.split(":")[0] for label in df_transposed.index]

# STEP 7: Identify group boundaries
group_boundaries_y = []
prev_group = None
for i, label in enumerate(df_transposed.index):
    group = label.split(":")[0]
    if group != prev_group:
        group_boundaries_y.append(i)
        prev_group = group
group_boundaries_y.append(len(df_transposed.index))

# STEP 8: Plot heatmap
fig_width = max(16, len(df_transposed.columns) * 0.5)
fig_height = max(22, len(df_transposed.index) * 0.5)
fig = plt.figure(figsize=(fig_width, fig_height))
gs = gridspec.GridSpec(1, 2, width_ratios=[20, 1], wspace=0.05)

ax = plt.subplot(gs[0])
cbar_ax = plt.subplot(gs[1])

# Convert mean values to numeric for heatmap coloring
heatmap_values = means.T
heatmap_values.rename(index=full_label_map, inplace=True)
heatmap_values *= 100

sns.heatmap(
    heatmap_values,
    annot=df_transposed,
    cmap='YlGnBu',
    fmt='',
    linewidths=.5,
    cbar_ax=cbar_ax,
    cbar_kws={'label': 'Claims Popularity: Share of Products with Claims (%)'},
    annot_kws={'fontsize': 15},
    ax=ax
)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=15)
cbar.ax.set_ylabel('Claims Popularity: Share of Products with Claims (%)', fontsize=15)

ax.set_yticklabels(cleaned_claim_names_y, rotation=0)
ax.set_xticklabels(df_transposed.columns, rotation=90)

# STEP 9: Draw group rectangles and vertical claim type labels
for i in range(len(group_boundaries_y) - 1):
    y_start = group_boundaries_y[i]
    y_end = group_boundaries_y[i + 1]
    group_name = group_names_y[group_boundaries_y[i]]

    rect = patches.Rectangle(
        (0, y_start),
        width=len(df_transposed.columns),
        height=y_end - y_start,
        linewidth=1.5,
        edgecolor='black',
        facecolor='none'
    )
    ax.add_patch(rect)

    ax.text(
        x=-4,
        y=(y_start + y_end - 1) / 2,
        s=group_name,
        ha='center',
        va='center',
        fontsize=16,
        fontweight='bold',
        rotation=90
    )

#ax.set_title('Heatmap of Claim Popularity by Category (with 95% CI)', fontsize=18)
ax.set_xlabel('Product Categories', fontweight='bold',fontsize=16)
ax.set_ylabel('Claims', fontweight='bold', fontsize=16, labelpad=60)
plt.tight_layout()

ax.tick_params(axis='x', labelsize=16)
ax.tick_params(axis='y', labelsize=16)

# Save figure
plt.tight_layout()
os.chdir(RESULT_PATH)
fig.savefig("Figure2_Heatmap_ClaimsPopularity_WithCI.svg", bbox_inches='tight')
fig.savefig("Figure2_Heatmap_ClaimsPopularity_WithCI.png", dpi=1200, bbox_inches='tight')
#fig.savefig("Figure2_Heatmap_ClaimsPopularity_WithCI.tiff", dpi=1200, bbox_inches='tight')
plt.show()



### Figure 3 Nutrition Profile Score with and without Claims

#### Figure3A. Discrepancy of Claims for Food Products

In [ ]:
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# 0. Use Food-only data
# ------------------------------------------------------------
df_nobeve = df.loc[df['NewCategory'] != 'Beverages'].copy()

# ------------------------------------------------------------
# 1. Define claim groups and readable labels
# ------------------------------------------------------------
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree',
        'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = {
    'LessCalorie': 'Low/No/Reduced Calorie',
    'NoAddedSugar': 'No Added Sugar',
    'SugarFree': 'Sugar Free',
    'LowSugar': 'Low/Reduced Sugar',
    'Diet': 'Diet/Light',
    'LessSodium': 'Low/No/Reduced Sodium',
    'LessCarb': 'Low/No/Reduced Carb',
    'LessFat': 'Low/No/Reduced Fat',
    'LessTransFat': 'Low/No/Reduced Trans Fat',
    'LessSatFat': 'Low/No/Reduced Saturated Fat',
    'LessChol': 'Low/No/Reduced Cholesterol',
    'LessGlycemic': 'Low/No/Reduced Glycemic',

    'PlusVitamin': 'Vitamin/Mineral Fortified',
    'HighProtein': 'High/Added Protein',
    'AddedCalcium': 'Added Calcium',
    'HighFiber': 'High/Added Fiber',

    'AllNatural': 'All Natural Product',
    'NoArtAdditives': 'No Added/Artificial Additives',
    'NoArtColourings': 'No Added/Artificial Colourings',
    'NoArtFlavourings': 'No Added/Artificial Flavourings',
    'NoArtPreservatives': 'No Added/Artificial Preservatives',
    'NoAdditivesPreservatives': 'No Additives/Preservatives',
    'GMOFree': 'GMO Free',
    'Organic': 'Organic',
    'Wholegrain': 'Whole Grain',

    'BrainNervSystem': 'Brain & Nervous System',
    'ImmuneSystem': 'Immune System',
    'Digestive': 'Digestive',
    'Probiotic': 'Probiotic/Prebiotic',
    'Antioxidant': 'Antioxidant',
    'WgtMuscle': 'Weight & Muscle Gain',
    'Cardiovascular': 'Cardiovascular',
    'BoneSkinHairEyeHealth': 'Bone/Skin/Nails & Hair/Eye Health'
}

claims = [c for grp in claim_groups.values() for c in grp]

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def get_significance_marker(p):
    if pd.isna(p):
        return ''
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

def assign_group(claim):
    for group, items in claim_groups.items():
        if claim in items:
            return group
    return 'Other'

# ------------------------------------------------------------
# 3. Compute Food-only results
# ------------------------------------------------------------
results = []

for claim in claims:
    if claim not in df_nobeve.columns:
        continue

    with_claim = df_nobeve.loc[df_nobeve[claim] == 1, 'np_score'].dropna()
    without_claim = df_nobeve.loc[df_nobeve[claim] == 0, 'np_score'].dropna()

    popularity = df_nobeve[claim].mean() * 100 if claim in df_nobeve.columns else np.nan

    mean_with = with_claim.mean() if len(with_claim) > 0 else np.nan
    mean_without = without_claim.mean() if len(without_claim) > 0 else np.nan
    direction = mean_with - mean_without if pd.notna(mean_with) and pd.notna(mean_without) else np.nan

    mw_stat, p_val = np.nan, np.nan
    if len(with_claim) > 0 and len(without_claim) > 0:
        try:
            mw_stat, p_val = mannwhitneyu(
                with_claim,
                without_claim,
                alternative='two-sided'
            )
        except ValueError:
            pass

    results.append({
        'Claim': claim,
        'Claim Label': claim_labels[claim],
        'With Claim': mean_with,
        'Without Claim': mean_without,
        'Mean Difference': direction,
        'Popularity': popularity,
        'MW Statistic': mw_stat,
        'p_value': p_val,
        'N With Claim': len(with_claim),
        'N Without Claim': len(without_claim)
    })

results_df_nobeve = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Add significance, colors, flags, groups
# ------------------------------------------------------------
results_df_nobeve['Group'] = results_df_nobeve['Claim'].apply(assign_group)
results_df_nobeve['Significance'] = results_df_nobeve['p_value'].apply(get_significance_marker)
results_df_nobeve['Direction'] = results_df_nobeve['With Claim'] - results_df_nobeve['Without Claim']

deep_palette = sns.color_palette('deep')
blue_deep = deep_palette[0]
orange_deep = deep_palette[1]

results_df_nobeve['Color'] = results_df_nobeve.apply(
    lambda row: orange_deep if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else
                blue_deep if pd.notna(row['Direction']) and row['Direction'] < 0 and row['p_value'] < 0.05 else
                'black',
    axis=1
)

results_df_nobeve['Label'] = results_df_nobeve.apply(
    lambda row: f"{row['Claim Label']} {row['Significance']}"
    if pd.notna(row['p_value']) and row['p_value'] < 0.05
    else row['Claim Label'],
    axis=1
)

results_df_nobeve['Flag'] = results_df_nobeve.apply(
    lambda row: True if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else False,
    axis=1
)

results_df = results_df_nobeve.copy()

# ------------------------------------------------------------
# 5. Save Food table
# ------------------------------------------------------------
table_to_save = results_df_nobeve[[
    'Group', 'Claim', 'Claim Label', 'With Claim', 'Without Claim',
    'Mean Difference', 'Popularity', 'MW Statistic', 'p_value',
    'Significance', 'N With Claim', 'N Without Claim', 'Flag'
]].copy()

csv_file = os.path.join(RESULT_PATH, "Figure3A_Discrepancy_Avg_NP_Score_Food_table.csv")
excel_file = os.path.join(RESULT_PATH, "Figure3A_Discrepancy_Avg_NP_Score_Food_table.xlsx")

table_to_save.to_csv(csv_file, index=False)
table_to_save.to_excel(excel_file, index=False)

# ------------------------------------------------------------
# 6. Melt for plotting
# ------------------------------------------------------------
results_melted = results_df_nobeve.melt(
    id_vars=['Claim', 'Popularity', 'p_value', 'Label', 'Color', 'Flag', 'Group'],
    value_vars=['With Claim', 'Without Claim'],
    var_name='Claim Status',
    value_name='Average NPM Score'
)

label_order = results_df_nobeve['Label'].tolist()
results_melted['Label'] = pd.Categorical(
    results_melted['Label'],
    categories=label_order,
    ordered=True
)

# ------------------------------------------------------------
# 7. Plot
# ------------------------------------------------------------
sns.set_theme(style="white")
fig = plt.figure(figsize=(9, 9))

scatter = sns.scatterplot(
    data=results_melted,
    x='Average NPM Score',
    y='Label',
    hue='Claim Status',
    size='Popularity',
    sizes=(50, 400),
    palette='deep',
    style='Claim Status',
    legend='brief'
)

ax = plt.gca()

# color y labels by direction/significance
label_color_map = dict(zip(results_df_nobeve['Label'], results_df_nobeve['Color']))
for tick in ax.get_yticklabels():
    txt = tick.get_text()
    if txt in label_color_map:
        tick.set_color(label_color_map[txt])

# ------------------------------------------------------------
# 8. Claim group spans and group labels
# ------------------------------------------------------------
label_to_group = dict(zip(results_df_nobeve['Label'], results_df_nobeve['Group']))
spans = []
prev_g, start_i = None, None

for i, lab in enumerate(label_order):
    g = label_to_group.get(lab, 'Other')
    if g != prev_g:
        if prev_g is not None:
            spans.append((start_i, i, prev_g))
        start_i = i
        prev_g = g
if prev_g is not None:
    spans.append((start_i, len(label_order), prev_g))

x0, x1 = ax.get_xlim()
for j, (s, e, g) in enumerate(spans):
    y0 = s - 0.5
    height = e - s
    if j != 0 and j != len(spans) - 1:
        ax.add_patch(
            patches.Rectangle(
                (x0, y0),
                width=(x1 - x0),
                height=height,
                fill=False,
                edgecolor='black',
                linewidth=1.2,
                zorder=1
            )
        )

xspan = x1 - x0
for (s, e, g) in spans:
    ax.text(
        x0 - 0.5 * xspan,
        (s + e - 1) / 2,
        g,
        ha='right',
        va='center',
        fontsize=12,
        rotation=90,
        fontweight='bold',
        color='black'
    )

# ------------------------------------------------------------
# 9. Vertical cutoff line and label
# ------------------------------------------------------------
cutoff_color = '#c44e52'
ax.axvline(
    x=4,
    color=cutoff_color,
    linestyle='--',
    linewidth=2,
    zorder=2
)

x0, x1 = ax.get_xlim()
ymin, ymax = ax.get_ylim()
yrange = ymax - ymin
x_cut = 4

ax.text(
    x_cut + 0.6,              
    ymin - 0.045 * yrange,    
    "NPM threshold     \n(≥4 for “less healthy”)",
    color=cutoff_color,
    fontsize=9,
    ha='right',         
    va='center',       
    zorder=3
)
# ------------------------------------------------------------
# 10. Formatting
# ------------------------------------------------------------
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.1f}'))

plt.xlabel("Average nutrient profile score", fontsize=12)
plt.ylabel("Claims", fontweight='bold', fontsize=12, labelpad=30)

ax.set_axisbelow(True)
ax.grid(
    True, which='both', axis='both',
    linestyle='--', linewidth=0.8, alpha=0.65, color='gray'
)

plt.tight_layout()

handles, labels = scatter.get_legend_handles_labels()
filtered = [(h, l) for h, l in zip(handles, labels) if l in ['With Claim', 'Without Claim']]
if filtered:
    h, l = zip(*filtered)
    l = ['With claim' if x == 'With Claim' else 'Without claim' for x in l]
    plt.legend(
        h, l,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.115),
        ncol=2,
        fontsize=12,
        frameon=True
    )

out = os.path.join(RESULT_PATH, "Figure3A_Discrepancy_Avg_NP_Score_Food.png")
if os.path.exists(out):
    os.remove(out)

plt.savefig(out, dpi=1200, bbox_inches='tight')
plt.show()

print(f"Saved figure: {out}")
print(f"Saved CSV: {csv_file}")
print(f"Saved Excel: {excel_file}")
print(table_to_save.head())


#### Figure3B. Components Contribution to Discrepancy

In [ ]:
# ============================================================
# Figure 3B (Food) 
# ============================================================

import matplotlib.ticker as mticker
import matplotlib.image as mpimg
from scipy.stats import mannwhitneyu

food_folder = os.path.join(RESULT_PATH, "Food")
os.makedirs(food_folder, exist_ok=True)

df_food = df.loc[df['NewCategory'] != 'Beverages'].copy()

deep_palette = sns.color_palette('deep')
blue_deep = deep_palette[0]
orange_deep = deep_palette[1]

a_components = ['energy_points', 'sat_fat_points', 'total_sugar_points', 'sodium_points']
c_components = ['fvn_points', 'fiber_points', 'protein_points']
all_components = a_components + c_components

component_label_map = {
    'energy_points': 'Energy',
    'sat_fat_points': 'Saturated Fat',
    'total_sugar_points': 'Total Sugar',
    'sodium_points': 'Sodium',
    'fvn_points': 'Fruit, Veg & Nuts',
    'fiber_points': 'Fiber',
    'protein_points': 'Protein'
}

claim_label = "Low/No/Reduced Trans Fat"

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def get_significance_marker(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return ''

def build_component_df(df_subset, claim):
    rows = []

    with_claim = df_subset.loc[df_subset[claim] == 1]
    without_claim = df_subset.loc[df_subset[claim] == 0]

    for comp in all_components:
        x = with_claim[comp].dropna()
        y = without_claim[comp].dropna()

        mean_diff = x.mean() - y.mean() if len(x) and len(y) else np.nan

        p_val = np.nan
        if len(x) > 0 and len(y) > 0:
            try:
                _, p_val = mannwhitneyu(x, y, alternative='two-sided')
            except:
                pass

        ci = np.nan
        if len(x) > 1 and len(y) > 1:
            se = np.sqrt(np.var(x, ddof=1)/len(x) + np.var(y, ddof=1)/len(y))
            ci = 1.96 * se

        contribution = mean_diff if comp in a_components else -mean_diff

        rows.append({
            'Component': comp,
            'Difference': mean_diff,
            'Contribution': contribution,
            'CI': ci,
            'Significance': get_significance_marker(p_val)
        })

    df_out = pd.DataFrame(rows)
    df_out['abs_contribution'] = df_out['Contribution'].abs()
    return df_out.sort_values('abs_contribution', ascending=False)

# ------------------------------------------------------------
# Build data
# ------------------------------------------------------------
panels = [
    ("Dairy & Eggs", df_food['NewCategory'] == "Dairy & Eggs"),
    ("Meals & Side Dishes", df_food['NewCategory'] == "Meals & Side Dishes")
]

comp_dfs = []

for cat, cond in panels:
    comp_df = build_component_df(df_food.loc[cond], 'LessTransFat')
    comp_dfs.append((cat, comp_df))

# ------------------------------------------------------------
# Common x-axis bound (tight)
# ------------------------------------------------------------
max_val = 0
for _, df_ in comp_dfs:
    for _, r in df_.iterrows():
        if pd.notna(r['Difference']):
            ci = r['CI'] if pd.notna(r['CI']) else 0
            max_val = max(max_val, abs(r['Difference']) + ci)

x_bound = max(0.5, round(max_val + 0.1, 1))

# ------------------------------------------------------------
# Plot 1×2 layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(8, 6))

for ax, (cat, comp_df) in zip(axes, comp_dfs):

    y_labels = [
        f"{component_label_map[c]} {s}".strip()
        for c, s in zip(comp_df['Component'], comp_df['Significance'])
    ]

    colors = [
        orange_deep if (
            (comp in a_components and diff > 0) or
            (comp in c_components and diff < 0)
        ) else blue_deep
        for comp, diff in zip(comp_df['Component'], comp_df['Difference'])
    ]

    ax.barh(
        range(len(comp_df)),
        comp_df['Difference'],
        xerr=comp_df['CI'],
        color=colors,
        capsize=4
    )

    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(y_labels, fontsize=8)

    ax.set_xlim(-x_bound, x_bound)
    ax.axvline(0, color='black', linewidth=1)

    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.tick_params(axis='x', labelsize=8)

    ax.set_title(f"{claim_label}\n({cat})", fontsize=9, pad=8)

    ax.invert_yaxis()
    ax.grid(True, linestyle='--', alpha=0.5)

# shared labels
axes[0].set_ylabel("Components of nutrient profile score", fontsize=8)
for ax in axes:
    ax.set_xlabel(  "Mean difference in component score\n(with claim − without claim)", fontsize=9, labelpad=10)

plt.tight_layout()

out_path = os.path.join(food_folder, "Figure3B_Food_Component_Discrepancy_Grid.png")
plt.savefig(out_path, dpi=1200, bbox_inches='tight')
plt.close()

print(f"✅ Figure 3B saved: {out_path}")



#### Figure 3. Combine Figure3A & Figure 3B

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

os.chdir(RESULT_PATH)

# --- inputs ---
fileA = "Figure3A_Discrepancy_Avg_NP_Score_Food.png"
fileB = os.path.join(food_folder,"Figure3B_Food_Component_Discrepancy_Grid.png")

out_path = os.path.join(
    RESULT_PATH,
    "Figure3_Discrepancy of Claims and Healthfulness of Food.png"
)

# Panel header text
text_a = "Nutrient profile scores by claim status: market-level patterns "
text_b = "Nutrient contributions to discrepancy within categories"

# --- open images ---
imgA = Image.open(fileA).convert("RGB")
imgB = Image.open(fileB).convert("RGB")

# --- match heights ---
target_height = max(imgA.height, imgB.height)
imgA = imgA.resize(
    (int(imgA.width * target_height / imgA.height), target_height),
    Image.LANCZOS
)
imgB = imgB.resize(
    (int(imgB.width * target_height / imgB.height), target_height),
    Image.LANCZOS
)

# --- top margin (reduced slightly) ---
top_margin = int(0.065 * target_height) 
combined_width = imgA.width + imgB.width
combined_height = target_height + top_margin

combined = Image.new("RGB", (combined_width, combined_height), (255, 255, 255))
combined.paste(imgA, (0, top_margin))
combined.paste(imgB, (imgA.width, top_margin))

draw = ImageDraw.Draw(combined)

# --- font loader ---
def load_font(candidates, size):
    for f in candidates:
        try:
            return ImageFont.truetype(f, size=size)
        except:
            continue
    return ImageFont.load_default()

# --- font sizes (smaller) ---
font_size = int(top_margin * 0.40)

font_bold = load_font(
    ["arialbd.ttf", "Arial Bold.ttf", "DejaVuSans-Bold.ttf"],
    font_size
)
font_reg = load_font(
    ["arial.ttf", "Arial.ttf", "DejaVuSans.ttf"],
    font_size
)

# --- placement (lower than before) ---
x_pad = int(0.012 * combined_width)
y_pad = int(0.35 * top_margin)   
gap   = int(0.008 * combined_width)

def draw_panel_header(x0, letter, header_text):
    # bold panel letter
    draw.text((x0, y_pad), letter, fill=(0, 0, 0), font=font_bold)

    # width of the letter to place text right after
    try:
        letter_w = draw.textlength(letter, font=font_bold)
    except:
        letter_w = font_bold.getsize(letter)[0]

    # regular header text
    draw.text(
        (x0 + letter_w + gap, y_pad),
        header_text,
        fill=(0, 0, 0),
        font=font_reg
    )

# --- draw headers ---
draw_panel_header(x_pad, "a", text_a)
draw_panel_header(imgA.width + x_pad, "b", text_b)

# --- save ---
combined.save(out_path, dpi=(900, 900), quality=100)
print(f"✅ Combined figure saved at: {out_path}")


## Figure 4 Beverages 

#### Figure4A. Discrepancy of Claims for Beverages

In [ ]:
# ============================================================
# Figure 4A: Beverages
# Average NP score for products with vs without claims
# Wilcoxon rank-sum test
# vertical cutoff line at NP score = 1
# ============================================================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu

# ------------------------------------------------------------
# 0. Use Beverage-only data
# ------------------------------------------------------------
df_beve = df.loc[df['NewCategory'] == 'Beverages'].copy()

# ------------------------------------------------------------
# 1. Define claim groups and readable labels
# ------------------------------------------------------------
claim_groups = {
    'Minus': [
        'LessCalorie', 'NoAddedSugar', 'SugarFree', 'LowSugar', 'Diet',
        'LessSodium', 'LessCarb', 'LessFat', 'LessTransFat', 'LessSatFat',
        'LessChol', 'LessGlycemic'
    ],
    'Plus': [
        'PlusVitamin', 'HighProtein', 'AddedCalcium', 'HighFiber'
    ],
    'Natural': [
        'AllNatural', 'NoArtAdditives', 'NoArtColourings', 'NoArtFlavourings',
        'NoArtPreservatives', 'NoAdditivesPreservatives', 'GMOFree',
        'Organic', 'Wholegrain'
    ],
    'Functional': [
        'BrainNervSystem', 'ImmuneSystem', 'Digestive', 'Probiotic',
        'Antioxidant', 'WgtMuscle', 'Cardiovascular', 'BoneSkinHairEyeHealth'
    ]
}

claim_labels = {
    'LessCalorie': 'Low/No/Reduced Calorie',
    'NoAddedSugar': 'No Added Sugar',
    'SugarFree': 'Sugar Free',
    'LowSugar': 'Low/Reduced Sugar',
    'Diet': 'Diet/Light',
    'LessSodium': 'Low/No/Reduced Sodium',
    'LessCarb': 'Low/No/Reduced Carb',
    'LessFat': 'Low/No/Reduced Fat',
    'LessTransFat': 'Low/No/Reduced Trans Fat',
    'LessSatFat': 'Low/No/Reduced Saturated Fat',
    'LessChol': 'Low/No/Reduced Cholesterol',
    'LessGlycemic': 'Low/No/Reduced Glycemic',

    'PlusVitamin': 'Vitamin/Mineral Fortified',
    'HighProtein': 'High/Added Protein',
    'AddedCalcium': 'Added Calcium',
    'HighFiber': 'High/Added Fiber',

    'AllNatural': 'All Natural Product',
    'NoArtAdditives': 'No Added/Artificial Additives',
    'NoArtColourings': 'No Added/Artificial Colourings',
    'NoArtFlavourings': 'No Added/Artificial Flavourings',
    'NoArtPreservatives': 'No Added/Artificial Preservatives',
    'NoAdditivesPreservatives': 'No Additives/Preservatives',
    'GMOFree': 'GMO Free',
    'Organic': 'Organic',
    'Wholegrain': 'Whole Grain',

    'BrainNervSystem': 'Brain & Nervous System',
    'ImmuneSystem': 'Immune System',
    'Digestive': 'Digestive',
    'Probiotic': 'Probiotic/Prebiotic',
    'Antioxidant': 'Antioxidant',
    'WgtMuscle': 'Weight & Muscle Gain',
    'Cardiovascular': 'Cardiovascular',
    'BoneSkinHairEyeHealth': 'Bone/Skin/Nails & Hair/Eye Health'
}

claims = [c for grp in claim_groups.values() for c in grp]

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def get_significance_marker(p):
    if pd.isna(p):
        return ''
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

def assign_group(claim):
    for group, items in claim_groups.items():
        if claim in items:
            return group
    return 'Other'

# ------------------------------------------------------------
# 3. Compute Beverage-only results
# ------------------------------------------------------------
results = []

for claim in claims:
    if claim not in df_beve.columns:
        continue

    with_claim = df_beve.loc[df_beve[claim] == 1, 'np_score'].dropna()
    without_claim = df_beve.loc[df_beve[claim] == 0, 'np_score'].dropna()

    popularity = df_beve[claim].mean() * 100 if claim in df_beve.columns else np.nan

    mean_with = with_claim.mean() if len(with_claim) > 0 else np.nan
    mean_without = without_claim.mean() if len(without_claim) > 0 else np.nan
    direction = mean_with - mean_without if pd.notna(mean_with) and pd.notna(mean_without) else np.nan

    mw_stat, p_val = np.nan, np.nan
    if len(with_claim) > 0 and len(without_claim) > 0:
        try:
            mw_stat, p_val = mannwhitneyu(
                with_claim,
                without_claim,
                alternative='two-sided'
            )
        except ValueError:
            pass

    results.append({
        'Claim': claim,
        'Claim Label': claim_labels[claim],
        'With Claim': mean_with,
        'Without Claim': mean_without,
        'Mean Difference': direction,
        'Popularity': popularity,
        'MW Statistic': mw_stat,
        'p_value': p_val,
        'N With Claim': len(with_claim),
        'N Without Claim': len(without_claim)
    })

results_df_beve = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Add significance, colors, flags, groups
# ------------------------------------------------------------
results_df_beve['Group'] = results_df_beve['Claim'].apply(assign_group)
results_df_beve['Significance'] = results_df_beve['p_value'].apply(get_significance_marker)
results_df_beve['Direction'] = results_df_beve['With Claim'] - results_df_beve['Without Claim']

deep_palette = sns.color_palette('deep')
blue_deep = deep_palette[0]
orange_deep = deep_palette[1]

results_df_beve['Color'] = results_df_beve.apply(
    lambda row: orange_deep if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else
                blue_deep if pd.notna(row['Direction']) and row['Direction'] < 0 and row['p_value'] < 0.05 else
                'black',
    axis=1
)

results_df_beve['Label'] = results_df_beve.apply(
    lambda row: f"{row['Claim Label']} {row['Significance']}"
    if pd.notna(row['p_value']) and row['p_value'] < 0.05
    else row['Claim Label'],
    axis=1
)

results_df_beve['Flag'] = results_df_beve.apply(
    lambda row: True if pd.notna(row['Direction']) and row['Direction'] > 0 and row['p_value'] < 0.05 else False,
    axis=1
)

# ------------------------------------------------------------
# 5. Save Beverage table
# ------------------------------------------------------------
table_to_save = results_df_beve[[
    'Group', 'Claim', 'Claim Label', 'With Claim', 'Without Claim',
    'Mean Difference', 'Popularity', 'MW Statistic', 'p_value',
    'Significance', 'N With Claim', 'N Without Claim', 'Flag'
]].copy()

csv_file = os.path.join(RESULT_PATH, "Figure4A_Discrepancy_Avg_NP_Score_Beverages_table.xlsx".replace(".xlsx", ".csv"))
excel_file = os.path.join(RESULT_PATH, "Figure4A_Discrepancy_Avg_NP_Score_Beverages_table.xlsx")

table_to_save.to_csv(csv_file, index=False)
table_to_save.to_excel(excel_file, index=False)

# ------------------------------------------------------------
# 6. Melt for plotting
# ------------------------------------------------------------
results_melted = results_df_beve.melt(
    id_vars=['Claim', 'Popularity', 'p_value', 'Label', 'Color', 'Flag', 'Group'],
    value_vars=['With Claim', 'Without Claim'],
    var_name='Claim Status',
    value_name='Average NPM Score'
)

label_order = results_df_beve['Label'].tolist()
results_melted['Label'] = pd.Categorical(
    results_melted['Label'],
    categories=label_order,
    ordered=True
)

# ------------------------------------------------------------
# 7. Plot
# ------------------------------------------------------------
sns.set_theme(style="white")
fig = plt.figure(figsize=(9.2, 9.0))

scatter = sns.scatterplot(
    data=results_melted,
    x='Average NPM Score',
    y='Label',
    hue='Claim Status',
    size='Popularity',
    sizes=(50, 400),
    palette='deep',
    style='Claim Status',
    legend='brief'
)

ax = plt.gca()

# Expand right edge so cutoff line does not touch border
x0_tmp, x1_tmp = ax.get_xlim()
right_pad = max(0.12 * (x1_tmp - x0_tmp), 0.25)
ax.set_xlim(x0_tmp, x1_tmp + right_pad)

# color y labels
label_color_map = dict(zip(results_df_beve['Label'], results_df_beve['Color']))
for tick in ax.get_yticklabels():
    txt = tick.get_text()
    if txt in label_color_map:
        tick.set_color(label_color_map[txt])

# ------------------------------------------------------------
# 8. Claim group spans and group labels
# ------------------------------------------------------------
label_to_group = dict(zip(results_df_beve['Label'], results_df_beve['Group']))
spans = []
prev_g, start_i = None, None

for i, lab in enumerate(label_order):
    g = label_to_group.get(lab, 'Other')
    if g != prev_g:
        if prev_g is not None:
            spans.append((start_i, i, prev_g))
        start_i = i
        prev_g = g
if prev_g is not None:
    spans.append((start_i, len(label_order), prev_g))

x0, x1 = ax.get_xlim()

for j, (s, e, g) in enumerate(spans):
    y0 = s - 0.5
    height = e - s
    if j != 0 and j != len(spans) - 1:
        ax.add_patch(
            patches.Rectangle(
                (x0, y0),
                width=(x1 - x0),
                height=height,
                fill=False,
                edgecolor='black',
                linewidth=1.2,
                zorder=1
            )
        )

xspan = x1 - x0
for (s, e, g) in spans:
    ax.text(
        x0 - 0.48 * xspan,
        (s + e - 1) / 2,
        g,
        ha='right',
        va='center',
        fontsize=12,
        rotation=90,
        fontweight='bold',
        color='black'
    )

# ------------------------------------------------------------
# 9. Vertical cutoff line and label
#    cutoff = 1 for beverages
# ------------------------------------------------------------
cutoff_color = '#c44e52'
ax.axvline(
    x=1,
    color=cutoff_color,
    linestyle='--',
    linewidth=2,
    zorder=2
)

x0, x1 = ax.get_xlim()
ymin, ymax = ax.get_ylim()
xspan = x1 - x0
yrange = ymax - ymin

x_cut = 1
x_text = min(x1 - 0.01 * xspan, x_cut + 0.18 * xspan)

y_text = ymin - 0.05 * yrange   

ax.text(
    x_text,
    y_text,
    "NPM threshold     \n(≥1 for “less healthy”)",
    color=cutoff_color,
    fontsize=9,
    ha='right',
    va='center',
    zorder=3
)

# ------------------------------------------------------------
# 10. Formatting
# ------------------------------------------------------------
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.1f}'))

plt.xlabel("Average nutrient profile score", fontsize=12)
plt.ylabel("Claims", fontweight='bold', fontsize=12, labelpad=30)

ax.set_axisbelow(True)
ax.grid(
    True, which='both', axis='both',
    linestyle='--', linewidth=0.8, alpha=0.55, color='gray'
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.16)

# legend
handles, labels = scatter.get_legend_handles_labels()
filtered = [(h, l) for h, l in zip(handles, labels) if l in ['With Claim', 'Without Claim']]
if filtered:
    h, l = zip(*filtered)
    l = ['With claim' if x == 'With Claim' else 'Without claim' for x in l]
    plt.legend(
        h, l,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.13),
        ncol=2,
        fontsize=12,
        frameon=True
    )

# save figure
out = os.path.join(RESULT_PATH, "Figure4A_Discrepancy_Avg_NP_Score_Beverages.png")
if os.path.exists(out):
    os.remove(out)

plt.savefig(out, dpi=1200, bbox_inches='tight')
plt.show()

print(f"Saved figure: {out}")
print(f"Saved CSV: {csv_file}")
print(f"Saved Excel: {excel_file}")
print(table_to_save.head())


#### Figure 4B: Component Contribution to Discpreancy for Discrepent Claims of Beverages

In [ ]:
# ============================================================
# Figure 4B (Beverages)
# SAME FORMAT as original
# ONLY updated: claims + subcategories selection
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.image as mpimg
from scipy.stats import mannwhitneyu

beve_folder = os.path.join(RESULT_PATH, "Beverages")
os.makedirs(beve_folder, exist_ok=True)

df_beve = df.loc[df['NewCategory'] == 'Beverages'].copy()

deep_palette = sns.color_palette('deep')
blue_deep = deep_palette[0]
orange_deep = deep_palette[1]

a_components = ['energy_points', 'sat_fat_points', 'total_sugar_points', 'sodium_points']
c_components = ['fvn_points', 'fiber_points', 'protein_points']
all_components = a_components + c_components

component_label_map = {
    'energy_points': 'Energy',
    'sat_fat_points': 'Saturated Fat',
    'total_sugar_points': 'Total Sugar',
    'sodium_points': 'Sodium',
    'fvn_points': 'Fruit, Veg & Nuts',
    'fiber_points': 'Fiber',
    'protein_points': 'Protein'
}

# local mapping to avoid any conflict with existing claim_labels object in memory
claim_label_map = {
    'LessSodium': 'Low/No/Reduced Sodium',
    'LessSatFat': 'Low/No/Reduced Saturated Fat'
}

def get_significance_marker(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''

def wrap_subtitle(text, max_len=40):
    parts = [x.strip() for x in text.split(';')]
    if len(parts) <= 1:
        return text

    line1 = []
    line2 = []
    current_len = 0

    for p in parts:
        add_len = len(p) + (2 if current_len > 0 else 0)
        if current_len + add_len <= max_len and len(line2) == 0:
            line1.append(p)
            current_len += add_len
        else:
            line2.append(p)

    if len(line2) == 0:
        return '; '.join(line1)
    return '; '.join(line1) + '\n' + '; '.join(line2)

def build_component_df_multi_group(df_main, claim, group_var, groups):
    component_rows = []

    for comp in all_components:
        group_diffs = []
        group_weights = []
        pooled_with_vals = []
        pooled_without_vals = []

        for g in groups:
            df_g = df_main.loc[df_main[group_var] == g].copy()

            with_vals = df_g.loc[df_g[claim] == 1, comp].dropna()
            without_vals = df_g.loc[df_g[claim] == 0, comp].dropna()

            if len(with_vals) == 0 or len(without_vals) == 0:
                continue

            diff_g = with_vals.mean() - without_vals.mean()
            weight_g = len(with_vals)

            group_diffs.append(diff_g)
            group_weights.append(weight_g)

            pooled_with_vals.extend(with_vals.tolist())
            pooled_without_vals.extend(without_vals.tolist())

        if len(group_diffs) == 0:
            diff = np.nan
        else:
            diff = np.average(group_diffs, weights=group_weights)

        pooled_with_vals = pd.Series(pooled_with_vals).dropna()
        pooled_without_vals = pd.Series(pooled_without_vals).dropna()

        p_val = np.nan
        if len(pooled_with_vals) > 0 and len(pooled_without_vals) > 0:
            try:
                _, p_val = mannwhitneyu(
                    pooled_with_vals,
                    pooled_without_vals,
                    alternative='two-sided'
                )
            except ValueError:
                p_val = np.nan

        if len(pooled_with_vals) > 1 and len(pooled_without_vals) > 1:
            se = np.sqrt(
                np.var(pooled_with_vals, ddof=1) / len(pooled_with_vals) +
                np.var(pooled_without_vals, ddof=1) / len(pooled_without_vals)
            )
            ci = 1.96 * se
        else:
            ci = np.nan

        contribution = diff if comp in a_components else -diff

        component_rows.append({
            'Component': comp,
            'Difference': diff,
            'Contribution': contribution,
            'CI': ci,
            'p_value': p_val,
            'Significance': get_significance_marker(p_val)
        })

    comp_df = pd.DataFrame(component_rows)
    comp_df['abs_contribution'] = comp_df['Contribution'].abs()
    comp_df = comp_df.sort_values('abs_contribution', ascending=False).reset_index(drop=True)
    return comp_df

def get_common_xbound(comp_dict):
    max_extent = 0
    for _, comp_df in comp_dict.items():
        for _, row in comp_df.iterrows():
            diff = row['Difference']
            ci = row['CI'] if pd.notna(row['CI']) else 0
            if pd.notna(diff):
                max_extent = max(max_extent, abs(diff) + ci)
    return max(0.5, math.ceil((max_extent + 0.10) / 0.5) * 0.5)

def draw_component_panel(comp_df, claim_title, subtitle, out_png, x_bound):
    y_labels = [
        f"{component_label_map[c]} {s}".strip()
        for c, s in zip(comp_df['Component'], comp_df['Significance'])
    ]

    colors = [
        orange_deep if (
            (comp in a_components and diff > 0) or
            (comp in c_components and diff < 0)
        ) else blue_deep
        for comp, diff in zip(comp_df['Component'], comp_df['Difference'])
    ]

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.barh(
        range(len(comp_df)),
        comp_df['Difference'],
        xerr=comp_df['CI'],
        color=colors,
        capsize=5
    )

    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(y_labels, fontsize=13)
    ax.tick_params(axis='x', labelsize=13)

    ax.set_xlabel(
        "Mean difference in component score\n(with claim − without claim)",
        fontsize=13
    )
    ax.set_ylabel("Components of nutrient profile score", fontsize=13)

    ax.set_title(f"{claim_title}\n{subtitle}", pad=10, fontsize=15)
    ax.invert_yaxis()

    ax.set_xlim(-x_bound, x_bound)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    ax.axvline(0, color='k', linewidth=1, alpha=0.7)

    ax.set_axisbelow(True)
    ax.grid(
        True, which='both', axis='both',
        linestyle='--', linewidth=0.8, alpha=0.65, color='gray'
    )

    plt.tight_layout(pad=0.6)
    plt.subplots_adjust(bottom=0.12)

    if os.path.exists(out_png):
        os.remove(out_png)
    plt.savefig(out_png, dpi=1200, bbox_inches='tight')
    plt.close(fig)

def combine_four_panels(panel_paths, out_png, fig_w=12.4, fig_h=10.0):
    fig, axes = plt.subplots(2, 2, figsize=(fig_w, fig_h))
    axes = axes.ravel()

    for ax, f in zip(axes, panel_paths):
        ax.imshow(mpimg.imread(f))
        ax.axis("off")

    for ax in axes[len(panel_paths):]:
        ax.axis("off")

    plt.subplots_adjust(
        left=0.04, right=0.985,
        top=0.985, bottom=0.05,
        wspace=-0.18,
        hspace=0.02
    )

    if os.path.exists(out_png):
        os.remove(out_png)
    plt.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.close(fig)

# ------------------------------------------------------------
# Final selected beverage claims and discrepant subcategories
# ONLY true discrepancy claims, split by subcategory
# ------------------------------------------------------------
beve_panels = [
    {
        'claim': 'LessSodium',
        'title': claim_label_map['LessSodium'],
        'subtitle': '(Carbonated Soft Drinks)',
        'groups': ['Carbonated Soft Drinks']
    },
    {
        'claim': 'LessSodium',
        'title': claim_label_map['LessSodium'],
        'subtitle': '(Fruit/Flavoured Still Drinks)',
        'groups': ['Fruit/Flavoured Still Drinks']
    },
    {
        'claim': 'LessSodium',
        'title': claim_label_map['LessSodium'],
        'subtitle': '(RTD Coffee & Tea)',
        'groups': ['RTD Coffee & Tea']
    },
    {
        'claim': 'LessSatFat',
        'title': claim_label_map['LessSatFat'],
        'subtitle': '(Fruit/Flavoured Still Drinks)',
        'groups': ['Fruit/Flavoured Still Drinks']
    }
]

# ------------------------------------------------------------
# Build component results (same logic/style as original)
# ------------------------------------------------------------
beve_comp_dict = {}
for i, spec in enumerate(beve_panels):
    beve_comp_dict[i] = build_component_df_multi_group(
        df_main=df_beve,
        claim=spec['claim'],
        group_var='NewSubCategory',
        groups=spec['groups']
    )

# ------------------------------------------------------------
# Common x-bound using original helper style
# ------------------------------------------------------------
beve_xbound = get_common_xbound(beve_comp_dict)
print(f"Common x-axis bound selected for Figure 4B: ±{beve_xbound:.2f}")

# ------------------------------------------------------------
# Draw panels
# ------------------------------------------------------------
beve_panel_paths = []

for i, spec in enumerate(beve_panels):
    comp_df = beve_comp_dict[i]
    subtitle = wrap_subtitle(spec['subtitle'], max_len=40)
    out_png = os.path.join(beve_folder, f"Figure4B_panel_{i+1}.png")

    draw_component_panel(
        comp_df=comp_df,
        claim_title=spec['title'],
        subtitle=subtitle,
        out_png=out_png,
        x_bound=beve_xbound
    )
    beve_panel_paths.append(out_png)

# ------------------------------------------------------------
# Combine panels
# ------------------------------------------------------------
beve_out = os.path.join(beve_folder, "Figure4B_Beverages_Component_Discrepancy_Grid.png")
combine_four_panels(beve_panel_paths, beve_out)

print(f"Saved Figure 4B: {beve_out}")


#### Figure 4: Combine Figure 4A and 4B

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

os.chdir(RESULT_PATH)

# --- inputs ---
fileA = "Figure4A_Discrepancy_Avg_NP_Score_Beverages.png"
fileB = os.path.join(beve_folder, "Figure4B_Beverages_Component_Discrepancy_Grid.png")

out_path = os.path.join(
    RESULT_PATH,
    "Figure4_Discrepancy of Claims and Healthfulness of Beverages.png"
)

# Panel header text
text_a = "Nutrient profile scores by claim status: market-level patterns"
text_b = "Nutrient contributions to discrepancy within categories"

# --- open images ---
imgA = Image.open(fileA).convert("RGB")
imgB = Image.open(fileB).convert("RGB")

# --- match heights ---
target_height = max(imgA.height, imgB.height)
imgA = imgA.resize(
    (int(imgA.width * target_height / imgA.height), target_height),
    Image.LANCZOS
)
imgB = imgB.resize(
    (int(imgB.width * target_height / imgB.height), target_height),
    Image.LANCZOS
)

# --- use SAME header-band style as Figure 3 ---
top_margin = int(0.065 * target_height)
combined_width = imgA.width + imgB.width
combined_height = target_height + top_margin

combined = Image.new("RGB", (combined_width, combined_height), (255, 255, 255))
combined.paste(imgA, (0, top_margin))
combined.paste(imgB, (imgA.width, top_margin))

draw = ImageDraw.Draw(combined)

# --- font loader ---
def load_font(candidates, size):
    for f in candidates:
        try:
            return ImageFont.truetype(f, size=size)
        except:
            continue
    return ImageFont.load_default()

# --- SAME font sizing logic as Figure 3 ---
font_size = int(top_margin * 0.40)

font_bold = load_font(
    ["arialbd.ttf", "Arial Bold.ttf", "DejaVuSans-Bold.ttf"],
    font_size
)
font_reg = load_font(
    ["arial.ttf", "Arial.ttf", "DejaVuSans.ttf"],
    font_size
)

# --- SAME placement logic as Figure 3 ---
x_pad = int(0.012 * combined_width)
y_pad = int(0.35 * top_margin)
gap = int(0.008 * combined_width)

def draw_panel_header(x0, letter, header_text):
    # bold panel letter
    draw.text((x0, y_pad), letter, fill=(0, 0, 0), font=font_bold)

    # width of the letter to place text right after
    try:
        letter_w = draw.textlength(letter, font=font_bold)
    except:
        letter_w = font_bold.getsize(letter)[0]

    # regular header text
    draw.text(
        (x0 + letter_w + gap, y_pad),
        header_text,
        fill=(0, 0, 0),
        font=font_reg
    )

# --- draw headers ---
draw_panel_header(x_pad, "a", text_a)
draw_panel_header(imgA.width + x_pad, "b", text_b)

# --- save ---
combined.save(out_path, dpi=(900, 900), quality=100)
print(f"✅ Combined figure saved at: {out_path}")